In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, roc_auc_score, log_loss, 
    classification_report, confusion_matrix, brier_score_loss
)
from sklearn.calibration import calibration_curve
import matplotlib.pyplot as plt

In [18]:
df = pd.read_csv('df.csv', index_col=0)

In [19]:
df.columns

Index(['match_id', 'player1', 'player2', 'PointServer', 'P1SetsWon',
       'P2SetsWon', 'P1GamesWon', 'P2GamesWon', 'IsTiebreak', 'P1Score',
       'P2Score', 'BestOf', 'MatchWinner', 'game_score_0-15',
       'game_score_0-30', 'game_score_0-40', 'game_score_15-0',
       'game_score_15-15', 'game_score_15-30', 'game_score_15-40',
       'game_score_30-0', 'game_score_30-15', 'game_score_30-30',
       'game_score_30-40', 'game_score_40-0', 'game_score_40-15',
       'game_score_40-30', 'game_score_40-40', 'game_score_40-AD',
       'game_score_AD-40'],
      dtype='object')

In [20]:
y = df['MatchWinner']
X = df.drop(columns=['MatchWinner', 'match_id','player1','player2', 'P1Score','P2Score'])

In [22]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### Logistic Regression

In [23]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

In [24]:
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"ROC AUC: {roc_auc_score(y_test, y_pred_proba):.4f}")
print(f"Log Loss: {log_loss(y_test, y_pred_proba):.4f}")
print(f"Brier Score: {brier_score_loss(y_test, y_pred_proba):.4f}")
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.7342
ROC AUC: 0.8142
Log Loss: 0.5218
Brier Score: 0.1753
              precision    recall  f1-score   support

           1       0.74      0.77      0.76      5005
           2       0.72      0.70      0.71      4336

    accuracy                           0.73      9341
   macro avg       0.73      0.73      0.73      9341
weighted avg       0.73      0.73      0.73      9341

[[3837 1168]
 [1315 3021]]


In [40]:
lr_coefs = pd.DataFrame({
    'Feature': X_train.columns,
    'Coefficient': model.coef_[0]
}).sort_values('Coefficient', key=abs, ascending=False)

print(lr_coefs)

             Feature  Coefficient
1          P1SetsWon    -1.302395
2          P2SetsWon     1.273931
18   game_score_40-0    -0.618627
4         P2GamesWon     0.476176
3         P1GamesWon    -0.471720
14   game_score_30-0    -0.421833
9    game_score_0-40     0.400471
19  game_score_40-15    -0.365850
23  game_score_AD-40    -0.339224
8    game_score_0-30     0.333514
13  game_score_15-40     0.308640
17  game_score_30-40     0.247898
10   game_score_15-0    -0.219574
22  game_score_40-AD     0.203139
15  game_score_30-15    -0.192481
20  game_score_40-30    -0.184601
6             BestOf     0.179740
12  game_score_15-30     0.178982
7    game_score_0-15     0.170150
16  game_score_30-30     0.056656
11  game_score_15-15     0.043915
5         IsTiebreak    -0.030447
0        PointServer    -0.023976
21  game_score_40-40    -0.012167


### Random Forest

In [29]:
rf = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
y_pred_proba = rf.predict_proba(X_test)[:, 1]

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"ROC AUC: {roc_auc_score(y_test, y_pred_proba):.4f}")
print(f"Log Loss: {log_loss(y_test, y_pred_proba):.4f}")
print(f"Brier Score: {brier_score_loss(y_test, y_pred_proba):.4f}")
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.7236
ROC AUC: 0.8077
Log Loss: 0.5773
Brier Score: 0.1952
              precision    recall  f1-score   support

           1       0.70      0.84      0.77      5005
           2       0.76      0.58      0.66      4336

    accuracy                           0.72      9341
   macro avg       0.73      0.71      0.71      9341
weighted avg       0.73      0.72      0.72      9341

[[4224  781]
 [1801 2535]]


In [39]:
rf_importances = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': rf.feature_importances_
}).sort_values('Importance', ascending=False)

print(rf_importances)

             Feature  Importance
2          P2SetsWon    0.376571
1          P1SetsWon    0.335861
4         P2GamesWon    0.120789
3         P1GamesWon    0.103165
6             BestOf    0.030656
18   game_score_40-0    0.004613
8    game_score_0-30    0.003406
0        PointServer    0.003287
14   game_score_30-0    0.003222
19  game_score_40-15    0.002847
13  game_score_15-40    0.002839
5         IsTiebreak    0.002233
9    game_score_0-40    0.002060
23  game_score_AD-40    0.002023
10   game_score_15-0    0.001263
12  game_score_15-30    0.001141
7    game_score_0-15    0.001006
15  game_score_30-15    0.000795
17  game_score_30-40    0.000566
22  game_score_40-AD    0.000543
21  game_score_40-40    0.000422
20  game_score_40-30    0.000283
16  game_score_30-30    0.000267
11  game_score_15-15    0.000142


### Gradient Boosting 

In [32]:
y_binary = (y == 1).astype(int)  # 1 if PointWinner==1 (P1 wins), else 0

X_train, X_test, y_train, y_test = train_test_split(X, y_binary, test_size=0.2, random_state=42)

# Now re-fit everything on the consistent y_train/y_test — logistic regression, random forest, and XGBoost
xgb_model = xgb.XGBClassifier(n_estimators=200, max_depth=4, random_state=42)
xgb_model.fit(X_train, y_train)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [33]:
xgb_model = xgb.XGBClassifier(n_estimators=200, max_depth=4, random_state=42)
xgb_model.fit(X_train, y_train)

y_pred = xgb_model.predict(X_test)
y_pred_proba = xgb_model.predict_proba(X_test)[:, 1]

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"ROC AUC: {roc_auc_score(y_test, y_pred_proba):.4f}")
print(f"Log Loss: {log_loss(y_test, y_pred_proba):.4f}")
print(f"Brier Score: {brier_score_loss(y_test, y_pred_proba):.4f}")
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.7395
ROC AUC: 0.8281
Log Loss: 0.4991
Brier Score: 0.1685
              precision    recall  f1-score   support

           0       0.73      0.71      0.72      4336
           1       0.75      0.77      0.76      5005

    accuracy                           0.74      9341
   macro avg       0.74      0.74      0.74      9341
weighted avg       0.74      0.74      0.74      9341

[[3057 1279]
 [1154 3851]]


In [41]:
xgb_importances = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': xgb_model.feature_importances_
}).sort_values('Importance', ascending=False)

print(xgb_importances)

             Feature  Importance
2          P2SetsWon    0.242935
1          P1SetsWon    0.231875
3         P1GamesWon    0.077759
4         P2GamesWon    0.061726
18   game_score_40-0    0.048293
6             BestOf    0.047682
19  game_score_40-15    0.028830
14   game_score_30-0    0.028104
23  game_score_AD-40    0.025608
8    game_score_0-30    0.021641
9    game_score_0-40    0.020795
13  game_score_15-40    0.019772
10   game_score_15-0    0.018402
15  game_score_30-15    0.016159
17  game_score_30-40    0.015165
21  game_score_40-40    0.014082
22  game_score_40-AD    0.013989
7    game_score_0-15    0.013761
0        PointServer    0.012434
20  game_score_40-30    0.012355
11  game_score_15-15    0.010114
12  game_score_15-30    0.009603
16  game_score_30-30    0.006229
5         IsTiebreak    0.002686


In [42]:
xgb_model.save_model('xgb_model.json')  # or .ubj for binary format

# To load the model later:
# xgb_model = xgb.XGBClassifier()
# xgb_model.load_model('xgb_model.json')